# BioRAG-X — Notebook 10: Evidence Selection → Grounded Generation → Citation Validation

## Research framing

This notebook closes the loop between **retrieval** and **trustworthy answer generation**.

The core pipeline is:

> **Retrieved candidates → evidence selection → context reconstruction → grounded generation → claim extraction → citation validation → regenerate / abstain**

The notebook also implements the controlled experiment:

1. **No Retrieval** — question → generator
2. **Oracle Retrieval** — question + gold passages → generator
3. **Actual Retrieval** — question + system-selected evidence → generator

The goal is not simply to maximize answer quality. We want to understand **how much retrieval helps, when retrieval hurts, whether the selected evidence is sufficient, and whether every generated claim is actually supported by cited biomedical evidence.**

### Important scientific rule

This notebook distinguishes **pipeline validation** from **scientific results**.

- A deterministic/mock generator is acceptable for validating data contracts and control flow.
- It must **not** be reported as evidence of biomedical LLM quality.
- Real generator results should only be reported when the configured model and dataset are actually available.


## Research questions

1. Does evidence-set selection improve answer quality relative to simply taking top-*k* passages?
2. Can context reconstruction preserve enough provenance to support claim-level citations?
3. How large is the gap between **Oracle Retrieval** and **Actual Retrieval**?
4. Does retrieved context improve generation, remain neutral, or actively hurt it?
5. How often does the system generate unsupported claims even when retrieval is strong?
6. Can citation validation reliably detect unsupported claims and trigger regeneration or abstention?
7. What is the cost of adding larger evidence sets and additional validation passes?
8. Which failures originate in retrieval/evidence selection versus generation?


## Metrics introduced here

### Evidence selection
- **Evidence Precision@K**
- **Evidence Recall@K**
- **Evidence Coverage**
- **Evidence Diversity**
- **Redundancy Rate**
- **Context Waste Ratio**
- **Provenance Completeness**

### Generation / grounding
- **Answer Correctness**
- **Answer Relevance**
- **Faithfulness / Grounding**
- **Unsupported Claim Rate**
- **Citation Precision**
- **Citation Recall**
- **Citation Support Rate**
- **Abstention Precision / Recall** when abstention labels are available

### Oracle decomposition
- **Retrieval Gap = Oracle − Actual**
- **Generation Gain = Actual − No Retrieval**
- **Retrieval Harm = Actual < No Retrieval**
- **Retrieval Help Rate**
- **Retrieval Harm Rate**


In [ ]:
from pathlib import Path
import json, math, re, time, statistics, hashlib
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple, Iterable

ROOT = Path("/mnt/data")
RUN_DIR = ROOT / "biorag_x_notebook10"
RUN_DIR.mkdir(exist_ok=True)

NOTEBOOKS = {
    "nb07": ROOT / "BioRAG-X_07_hybrid_retrieval_and_reranking.ipynb",
    "nb08": ROOT / "BioRAG-X_08_graph_and_pageindex.ipynb",
    "nb09": ROOT / "BioRAG-X_09_agentic_retrieval.ipynb",
}

for k, p in NOTEBOOKS.items():
    print(k, p.exists(), p)


## 1. Reproducible data contract

The generation layer must never receive an untraceable blob of text.

Each evidence item carries:
- `passage_id`
- `source_doc_id`
- `text`
- retrieval source/tool
- retrieval score
- rank
- provenance metadata
- optional section / parent context
- supporting graph edge(s), when applicable

This lets us reconstruct exactly **which evidence was shown to the generator** and later validate citations against that same evidence snapshot.


In [ ]:
@dataclass
class Evidence:
    passage_id: str
    text: str
    score: float = 0.0
    source: str = "unknown"
    rank: int = 0
    doc_id: Optional[str] = None
    section_id: Optional[str] = None
    parent_id: Optional[str] = None
    provenance: Dict[str, Any] = field(default_factory=dict)

@dataclass
class QueryRecord:
    question_id: str
    question: str
    answer_type: Optional[str] = None
    gold_answer: Optional[Any] = None
    gold_passage_ids: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class GeneratedClaim:
    claim_id: str
    text: str
    citations: List[str] = field(default_factory=list)
    supported: Optional[bool] = None
    support_score: Optional[float] = None

@dataclass
class GeneratedAnswer:
    text: str
    claims: List[GeneratedClaim]
    citations: List[str]
    mode: str
    raw: Dict[str, Any] = field(default_factory=dict)


## 2. Load the benchmark

The preferred source is the canonicalized BioASQ subset from the earlier notebooks. The loader searches for prior artifacts first and only falls back to lightweight demo data when the benchmark artifacts are unavailable.

The fallback is explicitly marked **DEMO ONLY**.


In [ ]:
import pandas as pd

def load_first_parquet(patterns):
    for pat in patterns:
        hits = sorted(ROOT.glob(pat))
        if hits:
            return hits[0]
    return None

candidate_paths = [
    ROOT / "biorag_x_canonical" / "qa.parquet",
    ROOT / "biorag_x_canonical" / "qa_gold.parquet",
    ROOT / "biorag_x" / "qa.parquet",
    ROOT / "biorag_x" / "qa_gold.parquet",
]
qa_path = next((p for p in candidate_paths if p.exists()), None)
if qa_path is None:
    qa_path = load_first_parquet(["**/*qa*.parquet", "**/*gold*.parquet"])

print("QA artifact:", qa_path)

if qa_path and qa_path.exists():
    qa_df = pd.read_parquet(qa_path)
    print("Loaded rows:", len(qa_df), "columns:", list(qa_df.columns))
else:
    qa_df = pd.DataFrame([
        {
            "question_id": "DEMO-1",
            "question": "What is the relationship between aspirin and platelet aggregation?",
            "gold_answer": "Aspirin inhibits platelet aggregation by irreversibly inhibiting cyclooxygenase-1 and reducing thromboxane A2 production.",
            "gold_passage_ids": ["P-DEMO-1", "P-DEMO-2"],
        },
        {
            "question_id": "DEMO-2",
            "question": "Which protein is associated with familial breast and ovarian cancer risk?",
            "gold_answer": "BRCA1 and BRCA2 are major genes associated with hereditary breast and ovarian cancer risk.",
            "gold_passage_ids": ["P-DEMO-3"],
        },
    ])
    print("WARNING: DEMO ONLY — no benchmark QA artifact was found.")


In [ ]:
def coalesce(row, names, default=None):
    for n in names:
        if n in row and pd.notna(row[n]):
            return row[n]
    return default

def normalize_gold_ids(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(v) for v in x]
    if isinstance(x, str):
        # support JSON lists and simple separators
        s = x.strip()
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return [str(v) for v in obj]
        except Exception:
            pass
        return [v.strip() for v in re.split(r"[;,|]", s) if v.strip()]
    return [str(x)]

records = []
for _, r in qa_df.iterrows():
    qid = str(coalesce(r, ["question_id","id","qid"], default=f"Q-{len(records)}"))
    q = str(coalesce(r, ["question","query","body"], default=""))
    gold = coalesce(r, ["gold_answer","answer","ideal_answer","reference_answer"], default="")
    gids = normalize_gold_ids(coalesce(r, ["gold_passage_ids","gold_passages","relevant_passage_ids"], default=[]))
    records.append(QueryRecord(
        question_id=qid,
        question=q,
        gold_answer=gold,
        gold_passage_ids=gids
    ))

queries = records
print("Query records:", len(queries))
print("First:", asdict(queries[0]) if queries else None)


## 3. Load or synthesize evidence candidates

The generator receives **evidence records, not raw retrieval outputs**.

When Notebook 07/08/09 artifacts contain retrieval outputs they should be adapted here. When they are unavailable, a deterministic demo candidate pool is constructed so that the notebook remains executable.

Again, demo outputs must not be treated as benchmark findings.


In [ ]:
def build_demo_evidence(record: QueryRecord) -> List[Evidence]:
    # Minimal, deterministic evidence solely for pipeline validation.
    q = record.question.lower()
    if "aspirin" in q:
        return [
            Evidence("P-DEMO-1", "Aspirin irreversibly inhibits cyclooxygenase-1 in platelets, reducing thromboxane A2 synthesis.", 0.91, "hybrid", 1, "D1", provenance={"source": "demo"}),
            Evidence("P-DEMO-2", "Reduced thromboxane A2 production decreases platelet activation and aggregation.", 0.87, "dense", 2, "D1", provenance={"source": "demo"}),
            Evidence("P-DEMO-4", "Aspirin also has analgesic and antipyretic effects.", 0.51, "bm25", 3, "D2", provenance={"source": "demo"}),
        ]
    if "brca" in q or "breast" in q or "ovarian" in q:
        return [
            Evidence("P-DEMO-3", "Pathogenic variants in BRCA1 and BRCA2 are strongly associated with hereditary breast and ovarian cancer susceptibility.", 0.94, "hybrid", 1, "D3", provenance={"source": "demo"}),
            Evidence("P-DEMO-5", "BRCA1 participates in DNA damage response and homologous recombination.", 0.73, "graph", 2, "D4", provenance={"source": "demo"}),
        ]
    return [
        Evidence("P-DEMO-G", "No benchmark-specific evidence was loaded; this is a demo evidence item.", 0.1, "demo", 1, "DEMO", provenance={"source":"demo"})
    ]

all_candidates = {r.question_id: build_demo_evidence(r) for r in queries}
print("Candidate pools:", len(all_candidates))


## 4. Evidence quality model

Reranking answers a different question from evidence selection.

A reranker asks:

> **How relevant is this one passage?**

Evidence selection asks:

> **Which *set* of passages is sufficient, diverse, non-redundant, and provenance-safe for answering this question?**

A practical set objective is:

**Utility = relevance + coverage + diversity + provenance − redundancy − token_cost**

We start with a deterministic heuristic so the selection policy itself can be benchmarked before introducing a learned/LLM selector.


In [ ]:
def tokenish(text: str) -> int:
    return max(1, len(re.findall(r"\w+", text)))

def lexical_overlap(a: str, b: str) -> float:
    A = set(re.findall(r"\w+", a.lower()))
    B = set(re.findall(r"\w+", b.lower()))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def select_evidence(
    question: str,
    candidates: List[Evidence],
    k: int = 4,
    redundancy_lambda: float = 0.35,
    diversity_lambda: float = 0.20,
    provenance_bonus: float = 0.05,
) -> List[Evidence]:
    selected = []
    remaining = candidates.copy()

    while remaining and len(selected) < k:
        best = None
        best_score = -1e9
        for ev in remaining:
            q_overlap = lexical_overlap(question, ev.text)
            max_red = max([lexical_overlap(ev.text, s.text) for s in selected], default=0.0)
            diversity = 1.0 - max_red
            provenance = 1.0 if ev.passage_id and ev.provenance else 0.0
            utility = (
                ev.score
                + 0.25 * q_overlap
                + diversity_lambda * diversity
                + provenance_bonus * provenance
                - redundancy_lambda * max_red
            )
            utility -= 0.001 * tokenish(ev.text)
            if utility > best_score:
                best_score, best = utility, ev
        selected.append(best)
        remaining.remove(best)

    for rank, ev in enumerate(selected, 1):
        ev.rank = rank
    return selected

selected_evidence = {
    r.question_id: select_evidence(r.question, all_candidates[r.question_id], k=4)
    for r in queries
}
for qid, evs in selected_evidence.items():
    print(qid, [e.passage_id for e in evs])


## 5. Gold evidence scoring

When gold passage IDs are available we can directly measure whether the evidence selector preserved the benchmark evidence.

These are *selection* metrics, not answer-generation metrics.


In [ ]:
def evidence_recall(selected: List[Evidence], gold_ids: List[str]) -> float:
    g = set(map(str, gold_ids))
    if not g:
        return math.nan
    s = {e.passage_id for e in selected}
    return len(s & g) / len(g)

def evidence_precision(selected: List[Evidence], gold_ids: List[str]) -> float:
    g = set(map(str, gold_ids))
    if not selected:
        return 0.0
    if not g:
        return math.nan
    return sum(e.passage_id in g for e in selected) / len(selected)

def context_waste_ratio(selected: List[Evidence], gold_ids: List[str]) -> float:
    g = set(map(str, gold_ids))
    if not selected:
        return 0.0
    waste = sum(e.passage_id not in g for e in selected)
    return waste / len(selected) if g else math.nan

selection_rows = []
for r in queries:
    evs = selected_evidence[r.question_id]
    selection_rows.append({
        "question_id": r.question_id,
        "evidence_precision": evidence_precision(evs, r.gold_passage_ids),
        "evidence_recall": evidence_recall(evs, r.gold_passage_ids),
        "context_waste_ratio": context_waste_ratio(evs, r.gold_passage_ids),
        "provenance_completeness": sum(bool(e.provenance) for e in evs) / len(evs) if evs else 0,
    })

selection_df = pd.DataFrame(selection_rows)
selection_df


## 6. Context reconstruction

Retrieval systems sometimes find a useful child passage while the answer requires the surrounding section, parent paragraph, table caption, or graph-supported context.

Context reconstruction therefore:
- preserves the original evidence identity,
- optionally attaches parent/section context,
- never invents provenance,
- records exactly what text was injected into the generator.

This becomes especially important for **parent-child chunking, late chunking, graph traversal, and PageIndex** outputs.


In [ ]:
def reconstruct_context(evidence: List[Evidence], max_tokens: int = 1200) -> Dict[str, Any]:
    ordered = sorted(evidence, key=lambda e: (e.rank or 999, -e.score))
    blocks = []
    used = 0
    source_ids = []

    for ev in ordered:
        t = tokenish(ev.text)
        if used + t > max_tokens:
            break
        blocks.append({
            "passage_id": ev.passage_id,
            "doc_id": ev.doc_id,
            "section_id": ev.section_id,
            "text": ev.text,
            "source": ev.source,
            "rank": ev.rank,
            "score": ev.score,
            "provenance": ev.provenance,
        })
        used += t
        source_ids.append(ev.passage_id)

    return {
        "blocks": blocks,
        "passage_ids": source_ids,
        "approx_tokens": used,
        "context_text": "\n\n".join(
            f"[{b['passage_id']}] {b['text']}" for b in blocks
        ),
    }

contexts = {
    qid: reconstruct_context(evs)
    for qid, evs in selected_evidence.items()
}
print(contexts[queries[0].question_id])


## 7. Generator contract

A production generator should receive a structured prompt and return structured output.

### Required output fields
- answer
- claims
- citation IDs attached to each claim
- optional confidence / abstention reason

The adapter below is deliberately model-agnostic. It can be connected later to an open biomedical instruction model, a hosted API, or an enterprise model endpoint.

The deterministic fallback is only for integration testing.


In [ ]:
class Generator:
    def __init__(self, mode="mock"):
        self.mode = mode

    def generate(self, question: str, context: str = "", mode: str = "actual") -> GeneratedAnswer:
        if self.mode == "mock":
            # Deterministic pipeline test; NOT a scientific LLM result.
            lines = [x.strip() for x in context.splitlines() if x.strip()]
            cited = []
            for line in lines:
                m = re.match(r"\[(.*?)\]", line)
                if m:
                    cited.append(m.group(1))
            text = (
                lines[0].split("] ", 1)[-1] if lines else
                "No retrieved evidence was supplied."
            )
            claims = [
                GeneratedClaim("C1", text, cited[:1], supported=None, support_score=None)
            ]
            return GeneratedAnswer(text=text, claims=claims, citations=cited, mode=mode)
        raise NotImplementedError("Connect a real LLM adapter before scientific generation experiments.")

generator = Generator(mode="mock")
print("Generator mode:", generator.mode)


## 8. Claim extraction

Citation validation should operate at **claim level**, not merely answer level.

Example:

> Aspirin inhibits COX-1 **[P1]** and lowers thromboxane A2 **[P2]**.

The validator evaluates whether each claim has sufficient evidence and whether the cited passage(s) support that claim.


In [ ]:
def extract_claims(answer_text: str, citations: List[str]) -> List[GeneratedClaim]:
    # Lightweight deterministic baseline. A production system can replace this with
    # sentence segmentation + claim decomposition + model-based atomic claim extraction.
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", answer_text) if s.strip()]
    claims = []
    for i, s in enumerate(sentences, 1):
        refs = re.findall(r"\[([^\]]+)\]", s)
        clean = re.sub(r"\s*\[[^\]]+\]", "", s).strip()
        if not refs:
            refs = citations[:1]
        claims.append(GeneratedClaim(f"C{i}", clean, refs))
    return claims

q0 = queries[0]
ga0 = generator.generate(q0.question, contexts[q0.question_id]["context_text"], mode="actual")
ga0.claims = extract_claims(ga0.text, ga0.citations)
ga0


## 9. Citation support model

We use a two-stage validator:

1. **Deterministic lexical support check** for a reproducible baseline.
2. Optional **NLI / entailment / LLM judge adapter** for stronger biomedical support checking.

The lexical baseline should be interpreted as a **diagnostic**, not a gold-standard measure of entailment.


In [ ]:
def citation_support_score(claim: str, cited_texts: List[str]) -> float:
    if not cited_texts:
        return 0.0
    claim_terms = set(re.findall(r"\b[a-zA-Z][a-zA-Z0-9-]{2,}\b", claim.lower()))
    if not claim_terms:
        return 0.0
    text_terms = set(re.findall(r"\b[a-zA-Z][a-zA-Z0-9-]{2,}\b", " ".join(cited_texts).lower()))
    return len(claim_terms & text_terms) / len(claim_terms)

def validate_claims(answer: GeneratedAnswer, evidence: List[Evidence], threshold: float = 0.35):
    by_id = {e.passage_id: e for e in evidence}
    for c in answer.claims:
        cited_texts = [by_id[cid].text for cid in c.citations if cid in by_id]
        c.support_score = citation_support_score(c.text, cited_texts)
        c.supported = c.support_score >= threshold
    return answer

def citation_metrics(answer: GeneratedAnswer):
    claims = answer.claims
    if not claims:
        return {"citation_precision": 0.0, "citation_recall": 0.0, "citation_support_rate": 0.0, "unsupported_claim_rate": 0.0}
    supported = sum(bool(c.supported) for c in claims)
    cited = sum(bool(c.citations) for c in claims)
    valid = sum(bool(c.citations) and bool(c.supported) for c in claims)
    return {
        "citation_precision": valid / cited if cited else 0.0,
        "citation_recall": valid / len(claims),
        "citation_support_rate": valid / len(claims),
        "unsupported_claim_rate": (len(claims) - supported) / len(claims),
    }

validated0 = validate_claims(ga0, selected_evidence[q0.question_id])
citation_metrics(validated0)


## 10. Sufficiency gate

Before generation, evidence should be classified as:

- **SUFFICIENT** — strong evidence coverage and provenance
- **WEAK** — some useful evidence but important support may be missing
- **INSUFFICIENT** — evidence is unlikely to support a grounded answer
- **CONFLICTING** — evidence contains material disagreement

The first implementation is rule-based so the behavior is observable. Later versions can learn the gate from benchmark labels or human judgments.


In [ ]:
def assess_sufficiency(question: str, evidence: List[Evidence], gold_ids: Optional[List[str]] = None) -> Dict[str, Any]:
    if not evidence:
        return {"status": "INSUFFICIENT", "score": 0.0, "reason": "no evidence"}

    mean_score = statistics.mean(e.score for e in evidence)
    prov = statistics.mean(1.0 if e.provenance else 0.0 for e in evidence)

    if gold_ids:
        recall = evidence_recall(evidence, gold_ids)
    else:
        recall = min(1.0, len(evidence) / 3.0)

    score = 0.55 * mean_score + 0.30 * recall + 0.15 * prov
    if score >= 0.75:
        status = "SUFFICIENT"
    elif score >= 0.50:
        status = "WEAK"
    else:
        status = "INSUFFICIENT"
    return {"status": status, "score": score, "reason": {"mean_score": mean_score, "coverage": recall, "provenance": prov}}


## 11. Regeneration / abstention policy

Citation validation changes the control flow:

```text
generate
  ↓
validate claims
  ↓
all claims supported?
  ├─ yes → answer
  └─ no
       ↓
  repair / regenerate
       ↓
  validate again
       ├─ supported → answer
       └─ unsupported → abstain
```

For production, the repair step may:
- remove unsupported claims,
- retrieve additional evidence,
- rewrite the answer,
- request stronger citations,
- or abstain.


In [ ]:
def repair_or_abstain(answer: GeneratedAnswer, evidence: List[Evidence], max_repairs: int = 1) -> Dict[str, Any]:
    metrics = citation_metrics(answer)
    if metrics["unsupported_claim_rate"] == 0:
        return {"decision": "ANSWER", "answer": answer, "repairs": 0}

    # Deterministic baseline: keep only supported claims and rebuild answer.
    supported_claims = [c for c in answer.claims if c.supported]
    if supported_claims:
        answer.claims = supported_claims
        answer.citations = sorted({cid for c in supported_claims for cid in c.citations})
        answer.text = " ".join(
            f"{c.text} " + " ".join(f"[{cid}]" for cid in c.citations)
            for c in supported_claims
        ).strip()
        return {"decision": "REPAIRED", "answer": answer, "repairs": 1}

    return {"decision": "ABSTAIN", "answer": answer, "repairs": 1}

def run_grounded_pipeline(record: QueryRecord, evidence: List[Evidence], generator: Generator):
    suff = assess_sufficiency(record.question, evidence, record.gold_passage_ids)
    if suff["status"] == "INSUFFICIENT":
        return {
            "sufficiency": suff,
            "decision": "ABSTAIN_BEFORE_GENERATION",
            "answer": None,
        }

    context = reconstruct_context(evidence)
    answer = generator.generate(record.question, context["context_text"], mode="actual")
    answer.claims = extract_claims(answer.text, answer.citations)
    answer = validate_claims(answer, evidence)

    result = repair_or_abstain(answer, evidence)
    result["sufficiency"] = suff
    result["context"] = context
    return result


## 12. Oracle experiment design

This is one of the most important experiments in the project.

For each benchmark question:

### A. No Retrieval
`question → generator`

Measures what the model can answer from its parametric knowledge / instructions alone.

### B. Oracle Retrieval
`question + gold passages → generator`

This isolates the generation ceiling given correct evidence.

### C. Actual Retrieval
`question + system-selected evidence → generator`

This measures the real system.

### Decomposition

**Retrieval Gap**

`Oracle − Actual`

A large gap indicates that retrieval/evidence selection is the bottleneck.

**Generation Gain**

`Actual − No Retrieval`

Positive = retrieval helps.  
Near zero = retrieval adds little.  
Negative = retrieval may be introducing noise, contradiction, or context overload.


In [ ]:
def build_oracle_evidence(record: QueryRecord, candidates: List[Evidence]) -> List[Evidence]:
    gold = set(map(str, record.gold_passage_ids))
    return [e for e in candidates if e.passage_id in gold]

def run_condition(record: QueryRecord, evidence: List[Evidence], condition: str):
    context = reconstruct_context(evidence)["context_text"] if evidence else ""
    ans = generator.generate(record.question, context, mode=condition)
    if evidence:
        ans.claims = extract_claims(ans.text, ans.citations)
        ans = validate_claims(ans, evidence)
    else:
        ans.claims = extract_claims(ans.text, ans.citations)
    return ans

experiment_rows = []
for record in queries:
    candidates = all_candidates[record.question_id]
    actual = selected_evidence[record.question_id]
    oracle = build_oracle_evidence(record, candidates)

    nr = run_condition(record, [], "no_retrieval")
    orc = run_condition(record, oracle, "oracle_retrieval")
    act = run_condition(record, actual, "actual_retrieval")

    experiment_rows.append({
        "question_id": record.question_id,
        "no_retrieval_support": citation_metrics(nr)["citation_support_rate"],
        "oracle_support": citation_metrics(orc)["citation_support_rate"],
        "actual_support": citation_metrics(act)["citation_support_rate"],
        "oracle_vs_actual_support_gap": citation_metrics(orc)["citation_support_rate"] - citation_metrics(act)["citation_support_rate"],
        "actual_vs_no_retrieval_gain": citation_metrics(act)["citation_support_rate"] - citation_metrics(nr)["citation_support_rate"],
    })

oracle_df = pd.DataFrame(experiment_rows)
oracle_df


## 13. Retrieval HELPED / NEUTRAL / HURT classification

For a production evaluation, the comparison should use a validated answer-quality metric such as:
- BioASQ exact / lenient / semantic metrics where applicable,
- biomedical expert scoring,
- groundedness / claim support,
- or a validated evaluator.

The classification logic itself is metric-agnostic.


In [ ]:
def classify_retrieval(no_ret_score: float, actual_score: float, eps: float = 1e-9) -> str:
    if actual_score > no_ret_score + eps:
        return "HELPED"
    if actual_score < no_ret_score - eps:
        return "HURT"
    return "NEUTRAL"

if len(oracle_df):
    oracle_df["retrieval_effect"] = [
        classify_retrieval(a, b)
        for a, b in zip(
            oracle_df["no_retrieval_support"],
            oracle_df["actual_support"]
        )
    ]

oracle_df


## 14. Context size and "Lost in the Middle"

More evidence is not automatically better.

We therefore sweep evidence-set size and context budget:

- `k ∈ {1, 2, 4, 8}`
- token budget ∈ small / medium / large

Track:
- answer quality
- claim support
- context waste
- latency
- tokens
- hallucination / unsupported claims

This is where the earlier retrieval stack connects directly to generation economics.


In [ ]:
def sweep_context_sizes(record: QueryRecord, candidate_pool: List[Evidence], ks=(1,2,4,8)):
    rows = []
    for k in ks:
        evs = select_evidence(record.question, candidate_pool, k=k)
        ctx = reconstruct_context(evs)
        ans = run_condition(record, evs, "actual")
        m = citation_metrics(ans)
        rows.append({
            "question_id": record.question_id,
            "k": k,
            "approx_tokens": ctx["approx_tokens"],
            "citation_support_rate": m["citation_support_rate"],
            "unsupported_claim_rate": m["unsupported_claim_rate"],
            "evidence_recall": evidence_recall(evs, record.gold_passage_ids),
        })
    return pd.DataFrame(rows)

sweep_df = pd.concat(
    [sweep_context_sizes(r, all_candidates[r.question_id]) for r in queries],
    ignore_index=True
) if queries else pd.DataFrame()

sweep_df.head()


## 15. Claim-level citation precision / recall

Interpretation:

- **Citation Precision** — of claims with citations, how many are actually supported?
- **Citation Recall** — of all answer claims, how many are supported by a valid citation?
- **Citation Support Rate** — fraction of claims that are both cited and supported.
- **Unsupported Claim Rate** — fraction of claims lacking adequate support.

A high answer score with a poor citation support rate is **not** a trustworthy biomedical RAG result.


In [ ]:
def aggregate_citation_metrics(answers: List[GeneratedAnswer]):
    if not answers:
        return {}
    ms = [citation_metrics(a) for a in answers]
    keys = ms[0].keys()
    return {k: statistics.mean(m[k] for m in ms) for k in keys}

actual_answers = []
for r in queries:
    evs = selected_evidence[r.question_id]
    a = run_condition(r, evs, "actual")
    actual_answers.append(a)

aggregate_citation_metrics(actual_answers)


## 16. Failure attribution

The same bad final answer can originate from different layers.

We tag failures using the BioRAG-X taxonomy:

- `F13` evidence incompleteness
- `F14` evidence conflict
- `F15` generation error
- `F16` hallucination
- `F17` citation failure
- `F18` should-have-abstained
- `F19` unnecessary retrieval
- `F20` latency / cost

The attribution record should also retain upstream retrieval diagnostics so that a generation problem is not incorrectly blamed on the retriever.


In [ ]:
def attribute_failure(record: QueryRecord, evidence: List[Evidence], answer: Optional[GeneratedAnswer]):
    tags = []
    suff = assess_sufficiency(record.question, evidence, record.gold_passage_ids)
    if suff["status"] == "INSUFFICIENT":
        tags.append("F13")
        tags.append("F18")
    if answer is not None:
        cm = citation_metrics(answer)
        if cm["unsupported_claim_rate"] > 0:
            tags.append("F17")
            tags.append("F16")
    if not evidence and answer is not None:
        tags.append("F19")
    return sorted(set(tags))

failure_rows = []
for r in queries:
    evs = selected_evidence[r.question_id]
    ans = run_condition(r, evs, "actual")
    failure_rows.append({
        "question_id": r.question_id,
        "failure_tags": attribute_failure(r, evs, ans),
    })

failure_df = pd.DataFrame(failure_rows)
failure_df


## 17. Production trace schema

A trustworthy RAG system should be observable end-to-end.

For every request, log a trace such as:

```text
query
 ├─ query_analysis
 ├─ retrieval_plan
 ├─ retrieval_calls
 ├─ candidate_pool
 ├─ reranking
 ├─ evidence_selection
 ├─ context_reconstruction
 ├─ generation
 ├─ claim_extraction
 ├─ citation_validation
 ├─ repair / regeneration
 └─ final decision
```

The trace should make it possible to answer:

> **"Why did the system make this claim, and exactly which evidence supported it?"**


In [ ]:
def make_trace(record: QueryRecord, selected: List[Evidence], answer: Optional[GeneratedAnswer], decision: str):
    return {
        "question_id": record.question_id,
        "question": record.question,
        "retrieval": [
            {
                "passage_id": e.passage_id,
                "score": e.score,
                "source": e.source,
                "rank": e.rank,
                "doc_id": e.doc_id,
                "section_id": e.section_id,
                "provenance": e.provenance,
            }
            for e in selected
        ],
        "evidence_set": [e.passage_id for e in selected],
        "generation": {
            "mode": answer.mode if answer else None,
            "claims": [asdict(c) for c in answer.claims] if answer else [],
            "citations": answer.citations if answer else [],
        },
        "final_decision": decision,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }

trace_examples = []
for r in queries[:3]:
    evs = selected_evidence[r.question_id]
    ans = run_condition(r, evs, "actual")
    trace_examples.append(make_trace(r, evs, ans, "ANSWER"))

trace_path = RUN_DIR / "trace_examples.json"
trace_path.write_text(json.dumps(trace_examples, indent=2), encoding="utf-8")
print(trace_path)


## 18. Ablation plan

Before claiming that the full evidence-generation stack is necessary, compare:

| Variant | Evidence Selection | Citation Validation | Repair | Expected research question |
|---|---:|---:|---:|---|
| Top-k only | No | No | No | Does selection matter? |
| Selected evidence | Yes | No | No | Does set optimization help? |
| + citation validator | Yes | Yes | No | Can unsupported claims be detected? |
| + repair | Yes | Yes | Yes | Can grounding failures be corrected? |
| + abstention | Yes | Yes | Yes | Can unsafe answers be withheld? |

The scientific result is the **delta between controlled variants**, not the absolute score of one configuration.


In [ ]:
def run_variant(record: QueryRecord, variant: str):
    candidates = all_candidates[record.question_id]

    if variant == "topk":
        evs = sorted(candidates, key=lambda e: e.score, reverse=True)[:4]
    else:
        evs = select_evidence(record.question, candidates, k=4)

    ans = run_condition(record, evs, "actual")

    if variant in {"validator", "repair", "abstain"}:
        ans = validate_claims(ans, evs)

    decision = "ANSWER"
    if variant in {"repair", "abstain"}:
        repaired = repair_or_abstain(ans, evs)
        decision = repaired["decision"]
        ans = repaired["answer"]

    m = citation_metrics(ans)
    return {
        "question_id": record.question_id,
        "variant": variant,
        "citation_support_rate": m["citation_support_rate"],
        "unsupported_claim_rate": m["unsupported_claim_rate"],
        "decision": decision,
    }

variants = ["topk", "selected", "validator", "repair", "abstain"]
ablation_df = pd.DataFrame(
    row
    for r in queries
    for row in [run_variant(r, v) for v in variants]
)
ablation_df


## 19. Save reproducible experiment artifacts

Save:
- selection metrics
- oracle decomposition
- context sweep
- failure tags
- ablation results
- generation/citation traces

This keeps the notebook useful as a benchmark artifact rather than a one-time demo.


In [ ]:
artifacts = {
    "selection_metrics.csv": selection_df,
    "oracle_decomposition.csv": oracle_df,
    "context_sweep.csv": sweep_df,
    "failure_attribution.csv": failure_df,
    "ablation.csv": ablation_df,
}

for filename, df in artifacts.items():
    path = RUN_DIR / filename
    df.to_csv(path, index=False)
    print(path)

manifest = {
    "notebook": "10_evidence_generation_and_citation",
    "benchmark_rows": len(queries),
    "generator_mode": generator.mode,
    "scientific_results_ready": generator.mode != "mock",
    "artifacts": [str(p) for p in RUN_DIR.glob("*")],
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))


## 20. Interpretation checklist before publishing results

Do **not** report a single "RAG score" without decomposing the failure.

For every experiment, ask:

1. Did the retriever find the gold evidence?
2. Did evidence selection preserve enough evidence?
3. Did context reconstruction preserve provenance?
4. Did retrieval help compared with no retrieval?
5. Did the oracle condition materially outperform actual retrieval?
6. Did generation produce unsupported claims?
7. Were unsupported claims caught by the validator?
8. Did repair actually improve grounding without destroying correctness?
9. Did abstention fire when evidence was insufficient?
10. How much latency/token cost did each safety layer add?

The central diagnosis is:

> **If Oracle ≫ Actual, fix retrieval/evidence selection.  
> If Actual ≫ No Retrieval but citation support is poor, fix grounding/validation.  
> If Actual ≈ No Retrieval, retrieval may be unnecessary or ineffective.  
> If Actual < No Retrieval, inspect retrieval harm and context noise.**


# Research conclusions to carry forward

Notebook 10 establishes the contract between **retrieval quality** and **answer trustworthiness**.

The complete BioRAG-X path is now:

```text
Query
  ↓
Adaptive Retrieval (Notebook 09)
  ↓
Candidate Evidence
  ↓
Evidence Selection
  ↓
Context Reconstruction
  ↓
Grounded Generation
  ↓
Claim Extraction
  ↓
Citation Validation
  ↓
Repair / Regenerate / Abstain
  ↓
Trace + Evaluation
```

## Handoff to Notebook 11

Notebook 11 should consolidate the entire project into a rigorous **evaluation + ablation framework**:

- retrieval benchmark
- BioASQ answer metrics
- RAG metrics
- citation metrics
- chunking regret
- routing regret
- recovery success
- retrieval help/harm
- ANN recall loss/speedup
- agent efficiency
- latency and cost
- statistical significance / confidence intervals
- full failure taxonomy
- final model/system leaderboard

Only after Notebook 11 should we move toward the production UI and deployment simulation.
